# Recursion and Backtracking

Notebook for **Chapter 6** of the *dsa* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/dsa/ch06.html

Tip: change a number, predict the output, then run the cell again.

## 6.1 Recursion from Zero: Base Case, Recursive Case and the Call Stack

In [ ]:
def fact(n, depth=0):
    pad = "  " * depth
    print(pad + "call fact(" + str(n) + ")")
    if n == 1:                      # base case
        print(pad + "base case, return 1")
        return 1
    smaller = fact(n - 1, depth + 1)   # same question, smaller input
    result = n * smaller               # combine
    print(pad + "fact(" + str(n) + ") = " + str(n) + " * " + str(smaller) + " = " + str(result))
    return result

assert fact(4) == 24

call fact(4)
  call fact(3)
    call fact(2)
      call fact(1)
      base case, return 1
    fact(2) = 2 * 1 = 2
  fact(3) = 3 * 2 = 6
fact(4) = 4 * 6 = 24


In [ ]:
import sys

def broken(n):
    return n * broken(n - 1)      # no base case

try:
    broken(5)
except RecursionError as e:
    print("caught:", type(e).__name__)
print("default limit:", sys.getrecursionlimit())

caught: RecursionError
default limit: 1000


In [ ]:
def total(nums, i=0):
    if i == len(nums):           # base case: nothing left
        return 0
    return nums[i] + total(nums, i + 1)

assert total([]) == 0
assert total([7]) == 7
assert total([3, 5, 2]) == 10
assert total([-4, 4, 1]) == 1
print(total([3, 5, 2]))

10


In [ ]:
calls = 0

def power(x, n):
    global calls
    calls += 1
    if n == 0:
        return 1
    half = power(x, n // 2)      # compute once, reuse
    if n % 2 == 0:
        return half * half
    return half * half * x

assert power(2, 10) == 1024
assert power(3, 0) == 1
assert power(5, 1) == 5
assert power(7, 13) == 7 ** 13
calls = 0
print(power(2, 1000) == 2 ** 1000, "calls:", calls)

True calls: 11


In [ ]:
def flatten(items):
    out = []
    for x in items:
        if isinstance(x, list):
            out.extend(flatten(x))   # smaller problem
        else:
            out.append(x)            # base: a plain number
    return out

assert flatten([]) == []
assert flatten([1, [2, [3]], 4]) == [1, 2, 3, 4]
assert flatten([[], [[]], 5]) == [5]
print(flatten([1, [2, [3, [4]]], [], 5]))

[1, 2, 3, 4, 5]


## 6.2 Recursion Trees, Complexity and a Memo Preview

In [ ]:
count = 0

def fib(n):
    global count
    count += 1
    if n < 2:
        return n
    return fib(n - 1) + fib(n - 2)

for n in [4, 5, 10, 20, 25]:
    count = 0
    value = fib(n)
    print("n =", n, " fib =", value, " calls =", count)

n = 4  fib = 3  calls = 9
n = 5  fib = 5  calls = 15
n = 10  fib = 55  calls = 177
n = 20  fib = 6765  calls = 21891
n = 25  fib = 75025  calls = 242785


In [ ]:
from functools import cache

body_runs = 0

@cache
def fib(n):
    global body_runs
    body_runs += 1               # counts real work, not cache hits
    if n < 2:
        return n
    return fib(n - 1) + fib(n - 2)

print(fib(25), "body runs:", body_runs)
print(fib(90), "body runs:", body_runs)

75025 body runs: 26
2880067194370816120 body runs: 91


In [ ]:
from functools import cache

@cache
def ways(n):
    if n <= 1:
        return 1
    return ways(n - 1) + ways(n - 2)

assert ways(0) == 1 and ways(1) == 1 and ways(2) == 2
assert ways(4) == 5
assert ways(10) == 89
print([ways(n) for n in range(1, 11)])

[1, 2, 3, 5, 8, 13, 21, 34, 55, 89]


In [ ]:
count = 0

def g(n):
    global count
    count += 1
    if n <= 1:
        return 1
    return g(n // 2) + g(n // 2) + n

for n in [1, 2, 8, 64, 1024]:
    count = 0
    g(n)
    print("n =", n, " calls =", count)

n = 1  calls = 1
n = 2  calls = 3
n = 8  calls = 15
n = 64  calls = 127
n = 1024  calls = 2047


## 6.3 The Backtracking Template and Subsets

In [ ]:
def solve(items):
    result = []        # all complete answers
    path = []          # the partial answer being built

    def backtrack(start):
        # 1. Record: is the path a complete answer? (here: every path counts)
        result.append(path[:])          # COPY the list, never the list itself
        # 2. Loop over the choices available at this step
        for i in range(start, len(items)):
            # 3. Prune: skip invalid choices here (none for plain subsets)
            path.append(items[i])       # choose
            backtrack(i + 1)            # explore, with a smaller set of choices
            path.pop()                  # unchoose (undo)

    backtrack(0)
    return result

assert solve([]) == [[]]
assert solve([5]) == [[], [5]]
assert len(solve(list(range(10)))) == 1024
print(solve([1, 2, 3]))

[[], [1], [1, 2], [1, 2, 3], [1, 3], [2], [2, 3], [3]]


In [ ]:
def wrong(items):
    result, path = [], []
    def bt(start):
        result.append(path)            # BUG: same list object each time
        for i in range(start, len(items)):
            path.append(items[i])
            bt(i + 1)
            path.pop()
    bt(0)
    return result

print(wrong([1, 2]))

[[], [], [], []]


In [ ]:
def count_subsets(nums, target):
    def bt(i, left):
        if left < 0:                     # prune: overshoot
            return 0
        if i == len(nums):
            return 1 if left == 0 else 0
        take = bt(i + 1, left - nums[i])
        skip = bt(i + 1, left)
        return take + skip
    return bt(0, target)

assert count_subsets([2, 3, 5], 5) == 2
assert count_subsets([1, 1, 1], 2) == 3
assert count_subsets([], 0) == 1
assert count_subsets([4], 3) == 0
print(count_subsets([2, 3, 5], 5), count_subsets([1, 2, 3, 4, 5], 5))

2 3


In [ ]:
def gen_parens(n):
    result = []
    path = []
    def bt(opened, closed):
        if len(path) == 2 * n:
            result.append("".join(path))
            return
        if opened < n:                    # may still open
            path.append("(")
            bt(opened + 1, closed)
            path.pop()
        if closed < opened:               # may close only an open one
            path.append(")")
            bt(opened, closed + 1)
            path.pop()
    bt(0, 0)
    return result

assert gen_parens(0) == [""]
assert gen_parens(1) == ["()"]
assert gen_parens(2) == ["(())", "()()"]
assert [len(gen_parens(k)) for k in range(1, 7)] == [1, 2, 5, 14, 42, 132]
print(gen_parens(3))

['((()))', '(()())', '(())()', '()(())', '()()()']


## 6.4 Permutations

In [ ]:
def permutations(nums):
    result, path = [], []
    used = [False] * len(nums)

    def bt():
        if len(path) == len(nums):          # complete answer
            result.append(path[:])
            return
        for i in range(len(nums)):          # every item is a candidate...
            if used[i]:                     # ...unless already placed
                continue
            used[i] = True                  # choose
            path.append(nums[i])
            bt()                            # explore
            path.pop()                      # unchoose
            used[i] = False

    bt()
    return result

assert permutations([]) == [[]]
assert permutations([1, 2]) == [[1, 2], [2, 1]]
assert len(permutations(list(range(6)))) == 720
print(permutations([1, 2, 3]))

[[1, 2, 3], [1, 3, 2], [2, 1, 3], [2, 3, 1], [3, 1, 2], [3, 2, 1]]


In [ ]:
def permute_swap(nums):
    result = []
    def bt(pos):
        if pos == len(nums):
            result.append(nums[:])
            return
        for j in range(pos, len(nums)):
            nums[pos], nums[j] = nums[j], nums[pos]     # choose
            bt(pos + 1)
            nums[pos], nums[j] = nums[j], nums[pos]     # unchoose
    bt(0)
    return result

got = permute_swap([1, 2, 3])
assert sorted(got) == [[1,2,3],[1,3,2],[2,1,3],[2,3,1],[3,1,2],[3,2,1]]
print(got)

[[1, 2, 3], [1, 3, 2], [2, 1, 3], [2, 3, 1], [3, 2, 1], [3, 1, 2]]


In [ ]:
def count_arrangements(n):
    used = [False] * (n + 1)
    def bt(pos):
        if pos > n:
            return 1
        total = 0
        for v in range(1, n + 1):
            if used[v]:
                continue
            if v % pos != 0 and pos % v != 0:    # prune
                continue
            used[v] = True
            total += bt(pos + 1)
            used[v] = False
        return total
    return bt(1)

assert count_arrangements(1) == 1
assert count_arrangements(2) == 2
assert count_arrangements(3) == 3
print([count_arrangements(n) for n in range(1, 9)])

[1, 2, 3, 8, 10, 36, 41, 132]


## 6.5 Combinations, Combination Sum and Pruning

In [ ]:
def combine(n, k):
    result, path = [], []
    def bt(start):
        if len(path) == k:                    # complete answer
            result.append(path[:])
            return
        # Prune: we still need (k - len(path)) items, so the last useful
        # start is n - (k - len(path)) + 1. Beyond that there are too few left.
        for i in range(start, n - (k - len(path)) + 2):
            path.append(i)
            bt(i + 1)
            path.pop()
    bt(1)
    return result

print(combine(4, 2))

[[1, 2], [1, 3], [1, 4], [2, 3], [2, 4], [3, 4]]


In [ ]:
from math import comb

def count_calls(n, k, prune):
    calls = 0
    found = 0
    path = []
    def bt(start):
        nonlocal calls, found
        calls += 1
        if len(path) == k:
            found += 1
            return
        stop = n - (k - len(path)) + 2 if prune else n + 1
        for i in range(start, stop):
            path.append(i)
            bt(i + 1)
            path.pop()
    bt(1)
    return found, calls

assert count_calls(4, 2, True)[0] == comb(4, 2) == 6
print("no prune:", count_calls(10, 8, False))
print("pruned:  ", count_calls(10, 8, True))

no prune: (45, 1013)
pruned:   (45, 165)


In [ ]:
def combination_sum(candidates, target):
    candidates = sorted(candidates)
    result, path = [], []
    def bt(start, left):
        if left == 0:
            result.append(path[:])
            return
        for i in range(start, len(candidates)):
            c = candidates[i]
            if c > left:               # prune: this and all later are too big
                break
            path.append(c)
            bt(i, left - c)            # i, not i + 1: reuse allowed
            path.pop()
    bt(0, target)
    return result

assert combination_sum([2, 3, 5], 5) == [[2, 3], [5]]
assert combination_sum([2], 1) == []
assert combination_sum([7], 7) == [[7]]
assert combination_sum([2, 3, 6, 7], 7) == [[2, 2, 3], [7]]
print(combination_sum([2, 3, 5], 8))

[[2, 2, 2, 2], [2, 3, 3], [3, 5]]


## 6.6 Handling Duplicates in the Input

In [ ]:
def subsets_unique(nums):
    nums = sorted(nums)                      # equal values become neighbours
    result, path = [], []
    def bt(start):
        result.append(path[:])
        for i in range(start, len(nums)):
            if i > start and nums[i] == nums[i - 1]:
                continue                     # same value already tried at this level
            path.append(nums[i])
            bt(i + 1)
            path.pop()
    bt(0)
    return result

assert subsets_unique([]) == [[]]
assert subsets_unique([3, 3, 3]) == [[], [3], [3, 3], [3, 3, 3]]
assert len(subsets_unique([4, 4, 1, 4])) == 8
print(subsets_unique([2, 1, 2]))

[[], [1], [1, 2], [1, 2, 2], [2], [2, 2]]


In [ ]:
def buggy(nums):
    nums = sorted(nums)
    result, path = [], []
    def bt(start):
        result.append(path[:])
        for i in range(start, len(nums)):
            if i > 0 and nums[i] == nums[i - 1]:     # BUG: should be i > start
                continue
            path.append(nums[i])
            bt(i + 1)
            path.pop()
    bt(0)
    return result

print(buggy([1, 2, 2]))

[[], [1], [1, 2], [2]]


In [ ]:
def permutations_unique(nums):
    nums = sorted(nums)
    result, path = [], []
    used = [False] * len(nums)
    def bt():
        if len(path) == len(nums):
            result.append(path[:])
            return
        for i in range(len(nums)):
            if used[i]:
                continue
            # equal twin to the left is still unused: placing this one would repeat an answer
            if i > 0 and nums[i] == nums[i - 1] and not used[i - 1]:
                continue
            used[i] = True
            path.append(nums[i])
            bt()
            path.pop()
            used[i] = False
    bt()
    return result

assert permutations_unique([1, 1, 2]) == [[1, 1, 2], [1, 2, 1], [2, 1, 1]]
assert permutations_unique([5, 5]) == [[5, 5]]
assert len(permutations_unique([1, 1, 2, 2, 3, 3])) == 90
print(permutations_unique([2, 1, 1]))

[[1, 1, 2], [1, 2, 1], [2, 1, 1]]


In [ ]:
def combination_sum2(nums, target):
    nums = sorted(nums)
    result, path = [], []
    def bt(start, left):
        if left == 0:
            result.append(path[:])
            return
        for i in range(start, len(nums)):
            if nums[i] > left:                          # too big: stop the loop
                break
            if i > start and nums[i] == nums[i - 1]:    # repeat at this level
                continue
            path.append(nums[i])
            bt(i + 1, left - nums[i])                   # once only
            path.pop()
    bt(0, target)
    return result

assert combination_sum2([1, 1, 2, 5], 4) == [[1, 1, 2]]
assert combination_sum2([2, 5, 2, 1, 2], 5) == [[1, 2, 2], [5]]
assert combination_sum2([3, 3, 3], 3) == [[3]]
assert combination_sum2([4], 3) == []
print(combination_sum2([10, 1, 2, 7, 6, 1, 5], 8))

[[1, 1, 6], [1, 2, 5], [1, 7], [2, 6]]


## 6.7 Backtracking on Strings: Letter Combinations, Palindrome Partitions and IP Addresses

In [ ]:
KEYS = {"2": "abc", "3": "def", "4": "ghi", "5": "jkl",
        "6": "mno", "7": "pqrs", "8": "tuv", "9": "wxyz"}

def letter_combinations(digits):
    if not digits:
        return []
    result, path = [], []
    def bt(i):
        if i == len(digits):
            result.append("".join(path))
            return
        for ch in KEYS[digits[i]]:
            path.append(ch)          # choose
            bt(i + 1)                # explore
            path.pop()               # unchoose
    bt(0)
    return result

assert letter_combinations("") == []
assert letter_combinations("2") == ["a", "b", "c"]
assert len(letter_combinations("79")) == 16
assert letter_combinations("23") == ["ad", "ae", "af", "bd", "be", "bf", "cd", "ce", "cf"]
print(letter_combinations("23"))

['ad', 'ae', 'af', 'bd', 'be', 'bf', 'cd', 'ce', 'cf']


In [ ]:
def palindrome_partitions(s):
    result, path = [], []
    def bt(start):
        if start == len(s):                       # all characters used
            result.append(path[:])
            return
        for end in range(start + 1, len(s) + 1):
            piece = s[start:end]
            if piece != piece[::-1]:              # prune: not a palindrome
                continue
            path.append(piece)
            bt(end)
            path.pop()
    bt(0)
    return result

assert palindrome_partitions("a") == [["a"]]
assert palindrome_partitions("aab") == [["a", "a", "b"], ["aa", "b"]]
assert len(palindrome_partitions("aaaa")) == 8
print(palindrome_partitions("aab"))
print(palindrome_partitions("abba"))

[['a', 'a', 'b'], ['aa', 'b']]
[['a', 'b', 'b', 'a'], ['a', 'bb', 'a'], ['abba']]


In [ ]:
def restore_ips(s):
    result, parts = [], []
    def valid(p):
        if len(p) > 1 and p[0] == "0":     # no leading zero
            return False
        return int(p) <= 255
    def bt(start):
        left_parts = 4 - len(parts)
        left_chars = len(s) - start
        if left_chars < left_parts or left_chars > 3 * left_parts:
            return                          # prune: impossible to finish
        if left_parts == 0:
            result.append(".".join(parts))
            return
        for size in (1, 2, 3):
            p = s[start:start + size]
            if len(p) < size or not valid(p):
                continue
            parts.append(p)
            bt(start + size)
            parts.pop()
    bt(0)
    return result

assert restore_ips("25525511135") == ["255.255.11.135", "255.255.111.35"]
assert restore_ips("0000") == ["0.0.0.0"]
assert restore_ips("111") == []
assert restore_ips("1111111111111") == []
print(restore_ips("101023"))

['1.0.10.23', '1.0.102.3', '10.1.0.23', '10.10.2.3', '101.0.2.3']


## 6.8 Grids and Boards: Word Search, N-Queens and Sudoku Awareness

In [ ]:
def exists(grid, word):
    rows, cols = len(grid), len(grid[0])
    moves = [(1, 0), (0, 1), (-1, 0), (0, -1)]       # down, right, up, left

    def dfs(r, c, k):
        # check bounds FIRST: a negative index would wrap around in Python
        if r < 0 or r >= rows or c < 0 or c >= cols:
            return False
        if grid[r][c] != word[k]:
            return False
        if k == len(word) - 1:
            return True
        saved = grid[r][c]
        grid[r][c] = "#"                              # choose: mark visited
        for dr, dc in moves:
            if dfs(r + dr, c + dc, k + 1):
                grid[r][c] = saved                    # leave the grid unchanged
                return True
        grid[r][c] = saved                            # unchoose
        return False

    return any(dfs(r, c, 0) for r in range(rows) for c in range(cols))

g = [list("CAR"), list("ATE"), list("DOG")]
assert exists(g, "CATE") and exists(g, "CAR") and exists(g, "DOG")
assert not exists(g, "CAB")
assert not exists([list("AB")], "ABA")
assert g == [list("CAR"), list("ATE"), list("DOG")]
print(exists(g, "CATE"), exists(g, "CAB"), exists([list("AB")], "ABA"))

True False False


In [ ]:
def exists_buggy(grid, word):
    rows, cols = len(grid), len(grid[0])
    moves = [(1, 0), (0, 1), (-1, 0), (0, -1)]
    def dfs(r, c, k):
        if r < 0 or r >= rows or c < 0 or c >= cols or grid[r][c] != word[k]:
            return False
        if k == len(word) - 1:
            return True
        grid[r][c] = "#"
        for dr, dc in moves:
            if dfs(r + dr, c + dc, k + 1):
                return True
        return False                    # BUG: forgot to restore the cell
    return any(dfs(r, c, 0) for r in range(rows) for c in range(cols))

print(exists_buggy([list("BBC"), list("BAA")], "AABC"))

False


In [ ]:
def solve_n_queens(n):
    solutions = []
    cols, diag1, diag2 = set(), set(), set()
    placed = []                                   # placed[r] = column of the queen in row r
    nodes = 0

    def bt(r):
        nonlocal nodes
        nodes += 1
        if r == n:
            solutions.append(placed[:])
            return
        for c in range(n):
            if c in cols or (r - c) in diag1 or (r + c) in diag2:
                continue                          # prune: attacked square
            cols.add(c); diag1.add(r - c); diag2.add(r + c)      # choose
            placed.append(c)
            bt(r + 1)                                             # explore
            placed.pop()
            cols.remove(c); diag1.remove(r - c); diag2.remove(r + c)   # unchoose

    bt(0)
    return solutions, nodes

def draw(cols_of_queens):
    n = len(cols_of_queens)
    return ["".join("Q" if c == q else "." for c in range(n)) for q in cols_of_queens]

counts = [len(solve_n_queens(n)[0]) for n in range(1, 9)]
assert counts == [1, 0, 0, 2, 10, 4, 40, 92]
sols, nodes = solve_n_queens(4)
assert sols == [[1, 3, 0, 2], [2, 0, 3, 1]]
print("counts n=1..8:", counts)
print("nodes visited for n=8:", solve_n_queens(8)[1])
for row in draw(sols[0]):
    print(row)

counts n=1..8: [1, 0, 0, 2, 10, 4, 40, 92]
nodes visited for n=8: 2057
.Q..
...Q
Q...
..Q.


In [ ]:
puzzle = ["53..7....", "6..195...", ".98....6.", "8...6...3", "4..8.3..1",
          "7...2...6", ".6....28.", "...419..5", "....8..79"]
board = [[0 if ch == "." else int(ch) for ch in row] for row in puzzle]

def ok(r, c, d):
    if any(board[r][j] == d for j in range(9)):            # row
        return False
    if any(board[i][c] == d for i in range(9)):            # column
        return False
    br, bc = 3 * (r // 3), 3 * (c // 3)                    # top-left of the 3x3 box
    return all(board[br + i][bc + j] != d for i in range(3) for j in range(3))

def solve():
    for r in range(9):
        for c in range(9):
            if board[r][c] == 0:                           # next empty cell
                for d in range(1, 10):
                    if ok(r, c, d):
                        board[r][c] = d                    # choose
                        if solve():                        # explore
                            return True
                        board[r][c] = 0                    # unchoose
                return False                               # no digit fits: backtrack
    return True                                            # no empty cell: solved

assert solve()
assert all(sorted(row) == list(range(1, 10)) for row in board)
assert all(sorted(board[i][c] for i in range(9)) == list(range(1, 10)) for c in range(9))
for row in board[:3]:
    print("".join(map(str, row)))

534678912
672195348
198342567


## 6.9 Converting Recursion to Iteration

In [ ]:
def flatten_iter(items):
    out = []
    stack = list(reversed(items))
    while stack:
        x = stack.pop()
        if isinstance(x, list):
            stack.extend(reversed(x))     # reversed, so the first item pops first
        else:
            out.append(x)
    return out

assert flatten_iter([]) == []
assert flatten_iter([1, [2, [3]], 4]) == [1, 2, 3, 4]
deep = [7]
for _ in range(5000):
    deep = [deep]
assert flatten_iter(deep) == [7]
print(flatten_iter([1, [2, [3]], 4]), "deep ok")

[1, 2, 3, 4] deep ok


In [ ]:
def subsets_mask(nums):
    n = len(nums)
    result = []
    for mask in range(1 << n):                      # 0 .. 2^n - 1
        result.append([nums[i] for i in range(n) if mask >> i & 1])
    return result

def subsets_double(nums):
    result = [[]]
    for x in nums:
        result += [s + [x] for s in result]         # copy every subset, add x
    return result

assert subsets_mask([]) == [[]] == subsets_double([])
a, b = subsets_mask([1, 2, 3]), subsets_double([1, 2, 3])
assert sorted(a) == sorted(b) and len(a) == 8
print(a)
print(b)

[[], [1], [2], [1, 2], [3], [1, 3], [2, 3], [1, 2, 3]]
[[], [1], [2], [1, 2], [3], [1, 3], [2, 3], [1, 2, 3]]


In [ ]:
def combination_sum_iter(candidates, target):
    candidates = sorted(candidates)
    result = []
    stack = [(0, target, ())]                       # (start, left, path)
    while stack:
        start, left, path = stack.pop()
        if left == 0:
            result.append(list(path))
            continue
        for i in range(start, len(candidates)):
            c = candidates[i]
            if c > left:                            # same prune as before
                break
            stack.append((i, left - c, path + (c,)))
    return result

assert sorted(combination_sum_iter([2, 3, 5], 8)) == [[2, 2, 2, 2], [2, 3, 3], [3, 5]]
assert combination_sum_iter([2, 3, 6, 7], 7) == [[7], [2, 2, 3]] or sorted(combination_sum_iter([2, 3, 6, 7], 7)) == [[2, 2, 3], [7]]
assert combination_sum_iter([4], 3) == []
assert len(combination_sum_iter([1], 30)) == 1
print(sorted(combination_sum_iter([2, 3, 5], 8)))

[[2, 2, 2, 2], [2, 3, 3], [3, 5]]


## Chapter recap

## Practice set

Twenty problems, graded from easy to hard. Try each one on paper before reading the answer. Statements are in my own words; the ideas are classic.

In [ ]:
def count_down(n):
    print(n)
    count_down(n - 1)

In [ ]:
def f(n):
    if n == 0:
        return
    print(n)
    f(n - 1)
    print(n)

f(3)

In [ ]:
def go(path, n):
    if len(path) == n:
        print(path)
        return
    for ch in "ab":
        go(path + ch, n)

go("", 2)

In [ ]:
def combination_sum(candidates, target):
    result, path = [], []
    def bt(start, left):
        if left == 0:
            result.append(path[:])
            return
        for i in range(start, len(candidates)):
            if candidates[i] > left:
                break
            path.append(candidates[i])
            bt(i + 1, left - candidates[i])      # <- look here
            path.pop()
    bt(0, target)
    return result

print(combination_sum([2, 3, 5], 6))

In [ ]:
def permutations(nums):
    result, path = [], []
    used = [False] * len(nums)
    def bt():
        if len(path) == len(nums):
            result.append(path[:])
            return
        for i in range(len(nums)):
            if used[i]:
                continue
            used[i] = True
            path.append(nums[i])
            bt()
            path.pop()
    bt()
    return result

print(permutations([1, 2, 3]))

In [ ]:
def no_adjacent_ones(n):
    result, path = [], []
    def bt():
        if len(path) == n:
            result.append("".join(path))
            return
        path.append("0")
        bt()
        path.pop()
        if not path or path[-1] != "1":          # prune
            path.append("1")
            bt()
            path.pop()
    bt()
    return result

assert no_adjacent_ones(3) == ["000", "001", "010", "100", "101"]
assert [len(no_adjacent_ones(k)) for k in range(1, 8)] == [2, 3, 5, 8, 13, 21, 34]
print(no_adjacent_ones(3))

['000', '001', '010', '100', '101']


In [ ]:
from functools import cache
from math import comb

@cache
def routes(r, c):
    if r == 1 or c == 1:
        return 1
    return routes(r - 1, c) + routes(r, c - 1)

assert routes(3, 3) == 6 == comb(4, 2)
assert routes(3, 7) == 28 == comb(8, 2)
assert routes(18, 18) == comb(34, 17)
print(routes(3, 3), routes(3, 7), routes(18, 18))

6 28 2333606220


In [ ]:
def case_variants(s):
    result, path = [], []
    def bt(i):
        if i == len(s):
            result.append("".join(path))
            return
        options = {s[i].lower(), s[i].upper()}       # one option for a digit
        for ch in sorted(options, reverse=True):     # lowercase first
            path.append(ch)
            bt(i + 1)
            path.pop()
    bt(0)
    return result

assert case_variants("a1b") == ["a1b", "a1B", "A1b", "A1B"]
assert case_variants("12") == ["12"]
assert len(case_variants("abcd")) == 16
print(case_variants("a1b"))

['a1b', 'a1B', 'A1b', 'A1B']


In [ ]:
from functools import cache

def target_sum_ways(nums, target):
    nums = tuple(nums)
    @cache
    def go(i, total):
        if i == len(nums):
            return 1 if total == target else 0
        return go(i + 1, total + nums[i]) + go(i + 1, total - nums[i])
    return go(0, 0)

assert target_sum_ways([1, 1, 1, 1, 1], 3) == 5
assert target_sum_ways([1], 1) == 1
assert target_sum_ways([1], 2) == 0
assert target_sum_ways([], 0) == 1
print(target_sum_ways([1, 1, 1, 1, 1], 3))

5


In [ ]:
from functools import cache

def can_split(nums):
    total = sum(nums)
    if total % 2:
        return False
    nums = tuple(sorted(nums, reverse=True))     # big numbers first prune sooner
    @cache
    def go(i, left):
        if left == 0:
            return True
        if i == len(nums) or left < 0:
            return False
        return go(i + 1, left - nums[i]) or go(i + 1, left)
    return go(0, total // 2)

assert can_split([1, 5, 11, 5]) is True
assert can_split([1, 2, 3, 5]) is False
assert can_split([]) is True
assert can_split([3]) is False
print(can_split([1, 5, 11, 5]), can_split([1, 2, 3, 5]))

True False


In [ ]:
def next_permutation(nums):
    i = len(nums) - 2
    while i >= 0 and nums[i] >= nums[i + 1]:     # find the pivot
        i -= 1
    if i >= 0:
        j = len(nums) - 1
        while nums[j] <= nums[i]:                # first larger value from the right
            j -= 1
        nums[i], nums[j] = nums[j], nums[i]
    nums[i + 1:] = reversed(nums[i + 1:])        # smallest suffix
    return nums

assert next_permutation([1, 2, 3]) == [1, 3, 2]
assert next_permutation([3, 2, 1]) == [1, 2, 3]
assert next_permutation([1, 1, 5]) == [1, 5, 1]
assert next_permutation([1, 3, 2]) == [2, 1, 3]
assert next_permutation([7]) == [7]
print(next_permutation([1, 3, 2]), next_permutation([1, 1, 5]))

[2, 1, 3] [1, 5, 1]


In [ ]:
def exists(grid, word):
    rows, cols = len(grid), len(grid[0])
    def dfs(r, c, k):
        if r >= rows or c >= cols:                # BUG: only the upper bounds
            return False
        if grid[r][c] != word[k]:
            return False
        if k == len(word) - 1:
            return True
        saved = grid[r][c]
        grid[r][c] = "#"
        for dr, dc in [(1, 0), (0, 1), (-1, 0), (0, -1)]:
            if dfs(r + dr, c + dc, k + 1):
                return True
        grid[r][c] = saved
        return False
    return any(dfs(r, c, 0) for r in range(rows) for c in range(cols))

print(exists([list("ABC"), list("DEF")], "AC"))

In [ ]:
def count_queens(n):
    full = (1 << n) - 1
    def bt(cols, d1, d2):
        if cols == full:
            return 1
        total = 0
        free = full & ~(cols | d1 | d2)          # bits of safe columns
        while free:
            bit = free & -free                   # lowest safe column
            free -= bit
            total += bt(cols | bit, ((d1 | bit) << 1) & full, (d2 | bit) >> 1)
        return total
    return bt(0, 0, 0)

assert count_queens(8) == 92
print([count_queens(n) for n in range(1, 10)])

[1, 0, 0, 2, 10, 4, 40, 92, 352]


In [ ]:
def add_signs(s, target):
    result = []
    def bt(start, total, expr):
        if start == len(s):
            if total == target:
                result.append(expr)
            return
        for end in range(start + 1, len(s) + 1):
            piece = s[start:end]
            if len(piece) > 1 and piece[0] == "0":      # leading zero: stop growing
                break
            v = int(piece)
            if start == 0:
                bt(end, v, piece)
            else:
                bt(end, total + v, expr + "+" + piece)
                bt(end, total - v, expr + "-" + piece)
    bt(0, 0, "")
    return result

assert add_signs("105", 5) == ["10-5"]
assert add_signs("123", 6) == ["1+2+3"]
assert add_signs("00", 0) == ["0+0", "0-0"]
assert add_signs("9", 8) == []
print(add_signs("105", 5), add_signs("00", 0))

['10-5'] ['0+0', '0-0']


## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/dsa/ch06.html